# Prerequisites

## Load Bronze data to dataframe

In [0]:
%sql
use catalog ecommerce;
create schema if not exists silver;


In [0]:
products_df = spark.read.table("bronze.products")
display(products_df.count())
products_df.limit(5).display()

In [0]:
products_df.printSchema()

# Transformations

## Drop Duplicates, product_id IS NOT NULL , validate price > 0 , validate cost > 0 

In [0]:
from pyspark.sql.functions import *
products_df = products_df.filter(col("product_id").isNotNull()).filter(col("price")>0).filter(col("cost")>0)
products_df = products_df.dropDuplicates()
display(products_df.count())
products_df.limit(5).display()

## Standardize product names  ,  product status.

In [0]:
from pyspark.sql.functions import *
products_df.select(col("product_status")).distinct().display()

In [0]:
products_df = products_df.withColumn(
    "product_name", initcap(trim(col("product_name")))
).withColumn("product_status", upper(trim(col("product_status"))))

## Validate price, code, category

In [0]:
valid_category = ["ELECTRONICS", "HOME", "FASHION", "SPORTS", "BEAUTY"]
products_df = (
    products_df.filter(upper(trim(col("category"))).isin(valid_category))
    .filter(col("price") >= 0)
    .filter(col("cost") >= 0)
)
display(products_df.count())
products_df.limit(5).display()

## Retain latest product information.

In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("product_id").orderBy(col("updated_at").desc())
products_df = (
    products_df.withColumn("rnk", row_number().over(window_spec))
    .filter(col("rnk") == 1)
    .drop("rnk")
)
display(products_df.count())
products_df.limit(5).display()

# Create Silver table

## Write to silver schema

In [0]:
products_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.products")

## Read silver.products table

In [0]:
spark.read.table("ecommerce.silver.products").limit(5).display()